# Modelos No Supervisados - Validación en Métodos de Clustering

Objetivo -> Análisis **exploratorio** + Análisis **confirmatorio**

## Análisis Exploratorio:

- Para entender patrones, tendencias, detectar anomalías
- Para resumir visualmente datos
- Para generar hipótesis que luego puedan ser confirmadas

## Análisis Confirmatorio:
Formas de validar/confirmar lo encontrado en el Análisis Exploratorio

- Corroborando con literatura
- Validando resultado mediante datos de test
- Validando la estabilidad de los resultados
- Verificando experimentalmente

## *Validando resultado mediante **datos de test***

Dividir los datos en Training y Test, antes del análisis.

Emplear Training para Exploratorio, y Test para Confirmatorio.

- Encontrar los K-clusters con training
- Luego, encontrar también K-clusters con test
- Con training, construir un modelo predictivo para K-etiquetas
- Predecir las etiquetas con los datos de test
- Comparar k-clusters de test con etiquetas predichas mediante métricas (Rand, Jaccard, Adjusted Rand Index)

***Rand Index (RI)***: mide la proporción de pares de puntos que están asignados igual o diferente en ambas particiones.

***Adjusted Rand Index (ARI)***: corrige el RI para el azar, de modo que un valor de 0 indica asignaciones aleatorias y 1 coincidencia perfecta.

***Jaccard Index***: similar a RI, pero considera solo los pares que están juntos en al menos una de las particiones.

| Métrica                       | Función                               | Qué mide                                                                               | Rango                             |
| ----------------------------- | ------------------------------------- | -------------------------------------------------------------------------------------- | --------------------------------- |
| **Rand Index**                | `rand_score`                          | Proporción de pares de muestras con la misma/diferente asignación en ambas particiones | 0–1                               |
| **Adjusted Rand Index (ARI)** | `adjusted_rand_score`                 | Igual que Rand, pero corrige el efecto del azar                                        | −1 a 1 (≈0 aleatorio, 1 perfecto) |
| **Jaccard Index**             | `jaccard_score(..., average='macro')` | Coincidencia promedio entre etiquetas (parecido a IoU)                                 | 0–1                               |


¿Cuándo usar macro o micro en el hiperparámetro average de Jaccard?

| Tipo de promedio      | Cómo calcula                                                                                                                                | Qué significa                                                         | Cuándo usar                                                                       |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------- | --------------------------------------------------------------------------------- |
| **`average='macro'`** | Calcula la métrica **por clase** y luego hace un **promedio simple** (todas las clases pesan igual)                                         | Evalúa el rendimiento **medio por clase**, sin importar su tamaño     | Si quieres que **todas las clases/clústeres cuenten igual**, incluso los pequeños |
| **`average='micro'`** | Combina **todas las predicciones** y **cuenta verdaderos positivos, falsos positivos y negativos globalmente** antes de calcular la métrica | Evalúa el rendimiento **global**, dando más peso a las clases grandes | Si te interesa el desempeño **ponderado por el tamaño del clúster**               |


In [ ]:
from sklearn.metrics import rand_score, adjusted_rand_score, jaccard_score
import numpy as np

true_labels = np.array([0, 0, 1, 1, 2, 2, 0, 0, 1, 1, 2])
pred_labels = np.array([1, 1, 0, 0, 2, 2, 1, 1, 0, 0, 2])

# Rand Index
ri = rand_score(true_labels, pred_labels)

# Adjusted Rand Index
ari = adjusted_rand_score(true_labels, pred_labels)

# Jaccard Index
ji = jaccard_score(true_labels, pred_labels, average='micro')

print(f"Rand Index: {ri:.3f}")
print(f"Adjusted Rand Index: {ari:.3f}")
print(f"Jaccard Index (micro): {ji:.3f}")

Rand Index: 1.000
Adjusted Rand Index: 1.000
Jaccard Index (micro): 0.158


## *Validando la **estabilidad** de los resultados*

*Principio de estabilidad:*

*"Si un hallazgo se repite consistentemente bajo pequeñas variaciones de los datos o del modelo, es más probable que sea real"*

Empleamos el 100% de los datos como training data, y mediante perturbaciones generamos nuevos datos de test.
- Si un patrón o clúster importante aparece en casi todas las repeticiones → es estable → probablemente es un hallazgo real.
- Si aparece solo ocasionalmente → probablemente es espurio o sensible al azar.

Otra forma de validar estabilidad es ralizando perturbaciones al proceso de aprendizaje (directamente a los modelos)

Perturbaciones más comunes:
- Subsampling: tomar una fracción del dataset original (por ejemplo, 80% de los datos al azar).
- Bootstrapping: muestrear (con reemplazo) del conjunto original (como en bagging).
- Agregar ruido: modificar levemente las características numéricas.
- Random corruptions: eliminar o distorsionar valores aleatoriamente.
- Random initializations / random tuning: cambiar las semillas o algunos hiperparámetros del modelo.

In [ ]:
#Ejemplo: Evaluar estabilidad de k-means con perturbaciones (bootstrapping + random init)

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt


# Configuración de la validación
K = 3                     # cantidad de clústeres
n_runs = 50               # cantidad de perturbaciones (repeticiones)
sample_frac = 0.8         # fracción de datos usada en cada bootstrap

# Guardamos los labels obtenidos en cada ejecución
all_labels = np.zeros((n_runs, len(X)))


# Repetir con perturbaciones
rng = np.random.default_rng(42)
for i in range(n_runs):
    # Bootstrapping: seleccionar una submuestra del 80% de los datos
    idx = rng.choice(len(X), size=int(sample_frac * len(X)), replace=False)
    X_boot = X[idx]

    # Entrenar K-Means con inicialización aleatoria
    kmeans = KMeans(n_clusters=K, n_init=1, init='random', random_state=None)
    kmeans.fit(X_boot)

    # Predecir clústeres (para comparar entre runs)
    labels = kmeans.predict(X)  # es una proyección geométrica, no es una predicción supervisada
    all_labels[i, :] = labels


# Medir estabilidad
# Calculamos el Adjusted Rand Index (ARI) entre todas las combinaciones de runs
stabilities = []
for i in range(n_runs):
    for j in range(i+1, n_runs):
        ari = adjusted_rand_score(all_labels[i, :], all_labels[j, :])
        stabilities.append(ari)

mean_stability = np.mean(stabilities)
print(f"Stability (mean Adjusted Rand Index): {mean_stability:.3f}")

# Visualización
plt.hist(stabilities, bins=15, color='skyblue', edgecolor='black')
plt.xlabel("Adjusted Rand Index entre runs")
plt.ylabel("Frecuencia")
plt.title("Distribución de la estabilidad de clustering")
plt.show()
